# 13 · BOUN data: exploration and attribution checks (Phase 6B, task 6B.1)

Exploration of the cleaned Boğaziçi (BOUN) works in `research/data/boun/` (built by `scripts/build_boun_clean.py`; data card in `audit/BOUN_DATA.md`).

- **Tables.** `boun_works.csv.gz` (one row per work, duplicates merged), `researchers.csv.gz` (id, name, number of works), `researcher_works.csv.gz` (researcher → work), and the raw OpenAlex records `works_openalex.jsonl.gz`.
- **Not in the repository.** Abstracts from Crossref / Semantic Scholar stay in the gitignored cache `data/v2/boun/`; they are merged below only if that cache exists.
- **Question.** Beyond the data card: how reliable is the BOUN attribution of authors, in large collaborations and in small works?

In [1]:
import gzip
import json
import re
import sys
from collections import Counter
from pathlib import Path

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

from src.boun_data import BOUN, MAX_BOUN_AUTHORS_PER_LARGE_WORK, names_boun

DATA = RESEARCH / "data" / "boun"
LIST_COLS = ["topics", "keywords", "concepts", "boun_author_ids", "author_ids", "duplicate_ids", "dq_strong", "dq_weak"]

In [2]:
works = pd.read_csv(DATA / "boun_works.csv.gz", keep_default_na=False)
for c in LIST_COLS:
    works[c] = works[c].map(json.loads)
people = pd.read_csv(DATA / "researchers.csv.gz", keep_default_na=False)
links = pd.read_csv(DATA / "researcher_works.csv.gz")
raw = {}
with gzip.open(DATA / "works_openalex.jsonl.gz", "rt") as f:
    for line in f:
        w = json.loads(line)
        raw[w["id"]] = w

ext_path = RESEARCH / "data" / "v2" / "boun" / "external_abstracts_clean.jsonl"
if ext_path.exists():
    ext = pd.read_json(ext_path, lines=True)[["id", "abstract_external"]]
    works = works.merge(ext, on="id", how="left")
    works["abstract_external"] = works["abstract_external"].fillna("")
else:
    works["abstract_external"] = ""
works["text_abstract"] = works["abstract"].where(works["abstract"] != "", works["abstract_external"])

print(json.loads((DATA / "build_stats.json").read_text()))
print(f"works {len(works):,} | researchers {len(people):,} | researcher-work rows {len(links):,} | raw records {len(raw):,}"
      f" | external abstracts merged: {ext_path.exists()}")

{'fetched': 8287, 'research_types': 7985, 'after_dedup': 7822, 'suspect_large_works': 6, 'merged_duplicates': 163, 'researchers_without_name': 0, 'researchers': 3648, 'researcher_work_rows': 17198}
works 7,822 | researchers 3,648 | researcher-work rows 17,198 | raw records 8,287 | external abstracts merged: True


## Overview

In [3]:
pd.concat({
    "type": works["type"].value_counts(),
    "abstract_source": works["abstract_source"].value_counts(),
    "language_flag": works["language_flag"].value_counts(),
}, axis=0).rename("works").to_frame()

works
type            article               5949
                conference-paper       772
                preprint               439
                book-chapter           349
                conference-abstract    203
                review                  36
                reference-entry         29
                book                    23
                report                  12
                dissertation             6
                data-paper               4
abstract_source openalex              5994
                none                  1237
                s2                     488
                crossref               103
language_flag   en                    7411
                tr                     245
                mixed                   81
                unknown                 67
                other                   18

In [4]:
pd.crosstab(works["publication_year"], works["abstract_source"], margins=True)

abstract_source,crossref,none,openalex,s2,All
publication_year,,,,,
2020,5,128,826,142,1101
2021,10,152,790,101,1053
2022,10,168,783,60,1021
2023,11,215,935,58,1219
2024,10,224,923,43,1200
2025,9,182,968,33,1192
2026,48,168,769,51,1036
All,103,1237,5994,488,7822


In [5]:
pd.crosstab(works["type"], works["abstract_source"], margins=True)

abstract_source,crossref,none,openalex,s2,All
type,,,,,
article,1,817,4705,426,5949
book,0,6,17,0,23
book-chapter,0,194,141,14,349
conference-abstract,0,3,200,0,203
conference-paper,0,143,588,41,772
data-paper,0,0,4,0,4
dissertation,0,0,6,0,6
preprint,102,56,277,4,439
reference-entry,0,14,15,0,29


In [6]:
works.assign(abstract_chars=works["text_abstract"].str.len()).groupby("abstract_source")["abstract_chars"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
abstract_source,,,,,,,,
crossref,103.0,1334.0,396.0,423.0,1052.0,1256.0,1655.0,2266.0
none,1237.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
openalex,5994.0,1310.0,723.0,1.0,972.0,1257.0,1578.0,20739.0
s2,488.0,1376.0,484.0,18.0,1037.0,1374.0,1708.0,2966.0


## Researchers

In [7]:
n = people["n_works"]
print(f"works per researcher: median {n.median():.0f}, mean {n.mean():.1f}; "
      f">=2 works: {(n >= 2).sum():,}, >=5: {(n >= 5).sum():,}, >=20: {(n >= 20).sum():,}")
people.sort_values("n_works", ascending=False).head(15)

works per researcher: median 1, mean 4.7; >=2 works: 1,672, >=5: 652, >=20: 115


,researcher_id,display_name,n_works
2909,https://openalex.org/A5107677778,E. Gülmez,479
2718,https://openalex.org/A5100759610,M. Kaya,375
120,https://openalex.org/A5004441810,S. Tekten,370
109,https://openalex.org/A5004024912,B. Akgün,324
1619,https://openalex.org/A5058983264,S. Iştın,265
2786,https://openalex.org/A5102874447,B. Gokturk,235
3024,https://openalex.org/A5111495737,V. Erkcan Özcan,222
7,https://openalex.org/A5000249500,Ismail Okan Atakisi,214
3057,https://openalex.org/A5114130509,I. O. Atakisi,213
3098,https://openalex.org/A5115589492,O. Kaya,186


In [8]:
large = set(works.loc[works["large_collaboration"], "id"])
small_works_per = links[~links["work_id"].isin(large)].groupby("researcher_id").size()
people["n_small_works"] = people["researcher_id"].map(small_works_per).fillna(0).astype(int)
people["collaboration_only"] = people["n_small_works"] == 0
print(f"collaboration-only researchers (every work has 100+ authors): {people['collaboration_only'].sum():,} of {len(people):,}")
people.groupby("collaboration_only")["n_works"].describe()

collaboration-only researchers (every work has 100+ authors): 311 of 3,648


,count,mean,std,min,25%,50%,75%,max
collaboration_only,,,,,,,,
False,3337.0,3.901708,13.328669,1.0,1.0,1.0,3.0,479.0
True,311.0,13.434084,45.577032,1.0,1.0,1.0,3.0,375.0


In [9]:
def profile(who: str, k: int = 20) -> pd.DataFrame:
    """Works of one researcher, by OpenAlex id or a case-insensitive name fragment."""
    hit = people[(people["researcher_id"] == who) | people["display_name"].str.contains(who, case=False, regex=False)]
    if hit.empty:
        raise KeyError(f"no researcher matches {who!r}")
    if len(hit) > 1:
        print(f"{len(hit)} matches; showing the first. Others:", hit["display_name"].tolist()[1:6])
    rid = hit.iloc[0]["researcher_id"]
    ids = set(links.loc[links["researcher_id"] == rid, "work_id"])
    print(hit.iloc[0]["display_name"], rid)
    return (works[works["id"].isin(ids)]
            .sort_values("first_date")[["first_date", "type", "title", "primary_topic", "abstract_source", "large_collaboration"]]
            .head(k))

# Example: the researcher with the median number of works among those with at least 5
example = people[people["n_works"] >= 5].sort_values("n_works").iloc[len(people[people["n_works"] >= 5]) // 2]
profile(example["researcher_id"])

Neslihan Yılmaz https://openalex.org/A5027972125


,first_date,type,title,primary_topic,abstract_source,large_collaboration
416,2020-05-13,article,"Money illusion, financial literacy and implications of self-perceptions","Financial Literacy, Pension, Retirement Analysis",openalex,False
921,2020-10-29,article,An Analysis of Spatial Dependence in Real Estate Prices,Housing Market and Economics,none,False
1051,2020-12-28,article,Spatial heterogeneity in Istanbul housing market: a geographically weighed approach,Housing Market and Economics,openalex,False
2397,2022-04-29,article,Is there an analyst (un)coverage premium?,Financial Markets and Investment Strategies,none,False
3478,2023-04-07,book-chapter,Analysis of Real Estate Prices Using Geospatial Data,Spatial and Panel Data Analysis,openalex,False
4888,2023-06-30,article,Analysis of Even Pricing in Real Estate Markets: Different Asset Types and Implications,Housing Market and Economics,openalex,False
5383,2024-11-16,article,"Corporate governance, blockholders, and financial distress: Global evidence",Corporate Finance and Governance,openalex,False
5496,2024-12-20,article,Social media disagreement and financial markets: A comparison of stocks and Bitcoin,Financial Markets and Investment Strategies,openalex,False
7157,2026-05-02,article,The dynamics of online social interactions and implications on stock market returns,Financial Markets and Investment Strategies,none,False


## How reliable is the BOUN attribution?

A researcher enters the data when OpenAlex links one of their authorships to BOUN (`I4405392`, or an institution whose lineage includes it). The build already drops unconfirmed BOUN authors from large works with more than `MAX_BOUN_AUTHORS_PER_LARGE_WORK` BOUN authors (`boun_attribution_suspect`); the checks below show why. Two checks:

1. **Large collaborations** (100+ authors). The full author lists come from single-work calls. A real BOUN group in ATLAS, CMS or BESIII is a handful of people per paper; many more BOUN authors on one paper means OpenAlex attached the BOUN affiliation string to the wrong authors.
2. **Small works** (< 100 authors). Does the authorship's raw affiliation text actually name Boğaziçi?

In [10]:
full_path = RESEARCH / "data" / "v2" / "boun" / "full_authors_named.jsonl"
if full_path.exists():  # BOUN authors per large work as OpenAlex returns them, before the build's correction
    full = {r["id"]: r for r in map(json.loads, full_path.open())}
    boun_per_large = pd.Series({i: len(r["boun_author_ids"]) for i, r in full.items() if i in large})
    print(boun_per_large.describe().round(1).to_dict())
    print(f"large works with more than {MAX_BOUN_AUTHORS_PER_LARGE_WORK} BOUN authors:", (boun_per_large > MAX_BOUN_AUTHORS_PER_LARGE_WORK).sum())
suspect_ids = works.loc[works["boun_attribution_suspect"], "id"]
works[works["boun_attribution_suspect"]][["first_date", "title"]].assign(
    boun_authors_openalex=[full[i]["n_authorships"] and len(full[i]["boun_author_ids"]) for i in suspect_ids] if full_path.exists() else None,
    total_authors=[full[i]["n_authorships"] for i in suspect_ids] if full_path.exists() else None,
    boun_authors_kept=works.loc[works["boun_attribution_suspect"], "boun_author_ids"].str.len())

{'count': 1004.0, 'mean': 7.0, 'std': 19.7, 'min': 0.0, '25%': 4.0, '50%': 6.0, '75%': 7.0, 'max': 370.0}
large works with more than 20 BOUN authors: 6


,first_date,title,boun_authors_openalex,total_authors,boun_authors_kept
711,2021-02-05,Measurements of Σ+ and Σ− time-like electromagnetic form factors for center-of-mass en...,247,492,3
1264,2021-04-29,Measurement of proton electromagnetic form factors in the time-like region using initi...,77,520,0
1523,2021-08-02,Measurement of cross section for e+e−→Ξ0Ξ¯0 near threshold,179,521,2
2066,2021-12-13,Search for new phenomena in three- or four-lepton events in pp collisions at s=13 TeV ...,370,2874,6
2501,2022-05-20,Measurement of the e+e−→Σ0Σ¯0 cross sections at center-of-mass energies from 2.3864 to...,186,491,1
3645,2023-06-15,Search for pair-production of vector-like quarks in pp collision events at s = 13 TeV ...,360,2881,7


In [11]:
# Researchers OpenAlex links to BOUN only through the suspect works (dropped by the build)
if full_path.exists():
    from_suspect = {a for i in suspect_ids for a in full[i]["boun_author_ids"]}
    elsewhere = set(links.loc[~links["work_id"].isin(set(suspect_ids)), "researcher_id"])
    print(f"BOUN authors on the {len(suspect_ids)} suspect works: {len(from_suspect):,}; "
          f"confirmed by another work and kept: {len(from_suspect & elsewhere):,}; dropped: {len(from_suspect - elsewhere):,}")

BOUN authors on the 6 suspect works: 1,003; confirmed by another work and kept: 12; dropped: 991


In [12]:
rows = []
for wid, w in raw.items():
    if len(w["authorships"]) >= 100:
        continue
    for a in w["authorships"]:
        aid = (a.get("author") or {}).get("id")
        if aid and any(BOUN == i.get("id") or BOUN in (i.get("lineage") or []) for i in a.get("institutions") or []):
            strings = a.get("raw_affiliation_strings") or []
            rows.append({"researcher_id": aid, "work_id": wid, "has_raw": bool(strings),
                         "raw_names_boun": any(names_boun(s) for s in strings)})
ev = pd.DataFrame(rows)
print(f"small-work BOUN authorships: {len(ev):,}; raw text present but without Boğaziçi: "
      f"{(ev['has_raw'] & ~ev['raw_names_boun']).sum():,}")
per = ev.groupby("researcher_id").agg(n=("work_id", "size"), supported=("raw_names_boun", "sum"), any_raw=("has_raw", "sum"))
weak_only = per[(per["supported"] == 0) & (per["any_raw"] > 0)]
print(f"researchers whose small-work evidence never names Boğaziçi in the raw text: {len(weak_only):,}")
people.set_index("researcher_id").loc[weak_only.index[:10], ["display_name", "n_works"]]

small-work BOUN authorships: 12,008; raw text present but without Boğaziçi: 78
researchers whose small-work evidence never names Boğaziçi in the raw text: 34


,display_name,n_works
researcher_id,,
https://openalex.org/A5003749280,Jeffrey Hawke,1
https://openalex.org/A5010744884,Selahattin Cansız,1
https://openalex.org/A5012733110,مهدی فاضلی,1
https://openalex.org/A5035249786,Dieu‐Donné Ouedraogo,1
https://openalex.org/A5037421064,Yalin Emek Çelik,1
https://openalex.org/A5042995866,Lygia Maria Copi,1
https://openalex.org/A5043106100,Çiğdem Gündüz-Demir,1
https://openalex.org/A5045941300,Sleda Poyraz,1
https://openalex.org/A5054700371,Enselme Zongo,1


In [13]:
# A few of those authorships, with the raw text OpenAlex matched to BOUN
sample = ev[ev["has_raw"] & ~ev["raw_names_boun"]].sample(min(5, int((ev["has_raw"] & ~ev["raw_names_boun"]).sum())), random_state=0)
for _, r in sample.iterrows():
    a = next(x for x in raw[r["work_id"]]["authorships"] if (x.get("author") or {}).get("id") == r["researcher_id"])
    print(a["author"]["display_name"], "|", raw[r["work_id"]]["title"][:70])
    print("   institutions:", [i.get("display_name") for i in a["institutions"]])
    print("   raw:", [s[:110] for s in a["raw_affiliation_strings"]][:2])

Çiğdem Gündüz-Demir | Histopathological Classification of Colon Tissue Images with Self-Supe
   institutions: ['Boğaziçi University']
   raw: ['Bilgisayar M&#x00FC;hendisli&#x011F;i B&#x00F6;l&#x00FC;m&#x00FC;, Ko&#x00E7; &#x00DC;niversitesi,&#x0130;stan']
Aylin Ulman | Some Biological Properties of Dolphinfish (Coryphaena hippurus) and it
   institutions: ["Compagnie Maritime d'Expertises (France)", 'Phenikaa (Vietnam)', 'France Énergies Marines', 'Feza Gürsey Institute']
   raw: ['Mersea Marine Consulting, Fethiye, Türkiye']
Andrzej Furman | Reproductive Isolation and Morphological Discrimination of Myotis myot
   institutions: ['Boğaziçi University']
   raw: ['Institute of Environmental Sciences, Bo—aziçi University, Hisar Campus, Bebek 34342, Istanbul, Turkey']
Göksu Gaml? | Asymmetries in the Production of Regular and Irregular Verbs
   institutions: ['Boğaziçi University']
   raw: ['Bo?aziçi University']
Sabri Gürses | Fyodor Mikhailovich Dostoevsky in Turkey: History of Translat

## Data-quality flags

In [14]:
print(Counter(f for fs in works["dq_strong"] for f in fs), Counter(f for fs in works["dq_weak"] for f in fs))
works[works["dq_strong"].str.len() > 0].sample(8, random_state=0)[["type", "title", "text_abstract", "dq_strong"]]

Counter({'title_like': 98, 'fragment': 24, 'boilerplate': 2, 'citation_string': 1}) Counter({'short_under_300': 194, 'availability_or_url': 99})


,type,title,text_abstract,dq_strong
3147,article,Yolsuzluğun İcadı,Silinecek,[title_like]
5525,article,"Interventions to Istanbul Studies II, Introduction Restoring the Fabric: Giving Voice ...",No abstract available,[title_like]
5526,article,Byzantine Studies and Istanbul s Greek-Speaking Community: The 1950s and the Tenth Int...,No abstract available,[title_like]
622,article,“Clickable” bacterial poly(γ-glutamic acid),"The controlled functionalization of bacterial γ-PGA is realized through sonication, so...",[title_like]
5645,article,Redox-responsive micellar nanoparticles using benzothiazole-disulfide terminated polym...,A modular approach to install functional chain-end groups through a redox-responsive d...,[title_like]
1679,book,Advances in Assessment and Modeling of Earthquake Loss,This open access book gathers contributions related to post-earthquake damage assessme...,[title_like]
591,article,Climate Change Effects on Agricultural Production: A Short Review,According to the Fifth Assessment Report (AR5) of the Intergovernmental Panel on Clima...,[title_like]
984,article,Guidelines for the use and interpretation of assays for monitoring autophagy (4th edit...,"autophagic responses. Here, we critically discuss current methods of assessing autopha...",[fragment]


In [15]:
works[works["language_flag"].isin(["tr", "mixed"])].sample(8, random_state=0)[["language_flag", "title", "text_abstract"]]

,language_flag,title,text_abstract
2817,tr,Asia Minor Mining Company’nin Finansal Sorunları ve Licese Madenindeki Faaliyetlerini ...,Osmanlı İmparatorluğu on dokuzuncu yüzyılın ikinci yarısında artan bir biçimde yeraltı...
2058,tr,Sevgi Soysal’ın Yenişehir’de Bir Öğle Vakti ve Yıldırım Bölge Kadınlar Koğuşu Metinler...,Sevgi Soysal basılan ilk öykü kitabı 1962 tarihli Tutkulu Perçem’de ve 1968 tarihli Ta...
3081,tr,Değişen İklimde Aşırı Hava Olaylarının İnsan Sağlığı Üzerinde Çok Boyutlu Etkileri ve ...,İklim değişikliği insan sağlığını doğrudan ve dolaylı olarak kısa veya uzun süreli etk...
5865,tr,Post-Ulusal Kimlik ve Geçmişle Yüzleşme Siyaseti: Almanya Örneği,Makalenin amacı Jürgen Habermas’ın müzakereci demokrasi anlayışının tamamlayıcı unsu...
3147,tr,Yolsuzluğun İcadı,Silinecek
2038,tr,Kritik Enerji Tesislerinin Deprem Risk Değerlendirmesinde Farklı Hasargörebilirlik Fon...,"Elektrik enerji tesisleri, deprem tehlikesinden dolayı hasar görme olasılığı yüksek kr..."
6420,mixed,"Geyik ve Güvercin Destanında Hayvan Temsili: Duygudaşlık, Mucize ve İhtida","Animal Representation in the Folktales of the Deer and the Dove: Empathy, Miracle and ..."
480,tr,Zamanın Sonu ve Bir İmparatorluğun Başlangıcı: Hicri Bininci Yılın Eşiğinde Bir İstanb...,Sultan II. Mehmet Konstantinapol’ü fethettiğinde tarihin bilinen en büyük imparatorluğ...


### Findings

- **Contents.** 7,822 works (5,949 articles, 772 conference papers, 439 preprints, 349 book chapters, 203 conference abstracts, others below 40). Abstract text: OpenAlex 5,994, Semantic Scholar 488, Crossref 103, none 1,237. Crossref helps almost only for preprints (102 of its 103); book chapters most often lack text (194 of 349).
- **Researchers.** 3,648 BOUN author ids, all named; median 1 work, 1,672 with at least 2, 652 with at least 5, 115 with at least 20. The most prolific ids are particle physicists in large collaborations (top: 479 works).
- **Author ids are sometimes split.** OpenAlex keeps "Ismail Okan Atakisi" (214 works) and "I. O. Atakisi" (213) as two ids, and two "O. Kaya" ids appear in the top 15. A profile per id can therefore be half a person; merging needs care (same initials are not the same person).
- **Misattributed collaborations (fixed in the build).** 6 large works (2 ATLAS, 4 BESIII; 491–2,881 authors) list 77–370 BOUN authors, against a median of 6 per large work. OpenAlex attached the BOUN affiliation string to the wrong authors. Of the 1,003 BOUN authors on these works, 12 are confirmed by another work and kept; **991 researchers who were BOUN only through these works are dropped** (4,639 → 3,648). Collaboration-only researchers fall from 1,302 to 311.
- **Small works.** 78 of 12,008 BOUN authorships (0.65%) have raw affiliation text that does not recognisably name Boğaziçi, and 34 researchers rest only on such authorships. The samples mix mangled spellings ("Bo?aziçi", "Bo—aziçi", a Russian address) with real OpenAlex errors (a Koç University address mapped to BOUN; a marine consultancy mapped to the Feza Gürsey Institute). They are kept and can be filtered with the check above.
- **Correction.** A first version of this check reported 483 such authorships and 107 researchers; most were BOUN addresses written with HTML entities (`Bo&#x011F;azi&#x00E7;i`) or split by PDF extraction (`Bogazic ¸i`), which `src.boun_data.names_boun` now handles.
- **Quality flags.** 121 works carry a strong "not an abstract" flag (98 `title_like`); the samples include placeholders such as "Silinecek" and "No abstract available". They are flagged, not removed.